# Part 1: Forward Mode Automatic Differentiation

Forward mode AD can simply be implemented by defining a class to represent [dual numbers](https://en.wikipedia.org/wiki/Dual_number) which hold the value and its derivative. The following skeleton defines a dual number and implements multiplication.

__Tasks:__

- Addition (`__add__`) is incomplete - can you finish it?
- Can you also implement division (`__truediv__`), subtraction (`__sub__`) and power (`__pow__`)?

In [22]:
import math

class DualNumber:
    def __init__(self, value, dvalue):
      self.value = value
      self.dvalue = dvalue

    def __str__(self):
      return str(self.value) + " + " + str(self.dvalue) + "ε"

    def __repr__(self):
      # для виводу результату
      return self.__str__()

    def __mul__(self, other):
      # правило добутку: (a + bε)(c + dε) = ac + (ad + bc)ε,  [ε² = 0]
      return DualNumber(self.value * other.value,
                        self.dvalue * other.value + other.dvalue * self.value)

    def __add__(self, other):
      # правило суми: (a + bε) + (c + dε) = (a + c) + (b + d)ε,  [ε² = 0]
      return DualNumber(self.value + other.value,
                        self.dvalue + other.dvalue)

    def __sub__(self, other):
      # правило різниці: (a + bε) - (c + dε) = (a - c) + (b - d)ε
      return DualNumber(self.value - other.value,
                        self.dvalue - other.dvalue)

    def __truediv__(self, other):
      # правило частки: (a + bε)/(c + dε) = a/c + ((bc - ad)/c^2)ε
      return DualNumber(self.value / other.value,
                        (self.dvalue * other.value - self.value * other.dvalue) / (other.value ** 2))

    def __pow__(self, other):
      # Загальний випадок u^v: d(u^v) = u^v·(v'·ln u + v·u'/u)
      # Якщо показник — константа (v' = 0), використовуємо степеневе правило
      # v·u^(v-1)·u', щоб не брати ln від недодатної основи.
      value = self.value ** other.value
      if other.dvalue == 0:
        dvalue = other.value * self.value ** (other.value - 1) * self.dvalue
      else:
        dvalue = value * (other.dvalue * math.log(self.value) + other.value * self.dvalue / self.value)

      return DualNumber(value, dvalue)




In [23]:
# Tests

DualNumber(1,0) + DualNumber(1,0) / DualNumber(1,0) - DualNumber(1,0)**DualNumber(1,0)


1.0 + 0.0ε

## Implementing math functions

We also need to implement some core math functions. Here's the sine function for a dual number:

In [24]:
def sin(x):
    return DualNumber(math.sin(x.value), math.cos(x.value)*x.dvalue)

__Task:__ can you implement the _cosine_ (`cos`), _tangent_ (`tan`), and _exponential_ (`exp`) functions in the code block below?

In [25]:
# TODO: implement additional math functions on dual numbers
# Загальне правило (ланцюгове правило у forward-режимі):
#   f(a + bε) = f(a) + f'(a)·b·ε

def cos(x):
    # (cos(u))' = -sin(u) * u'
    return DualNumber(math.cos(x.value), -math.sin(x.value) * x.dvalue)

def tan(x):
    # (tan(u))' = u' / cos(u) ** 2
    return DualNumber(math.tan(x.value), x.dvalue / (math.cos(x.value) ** 2))

def exp(x):
    # (exp(u))' = exp(u) * u'
    e = math.exp(x.value)
    return DualNumber(e, e * x.dvalue)

In [26]:
# Tests
assert cos(DualNumber(0,0)).value == 1
assert tan(DualNumber(0,0)).value == 0
assert exp(DualNumber(0,0)).value == 1


## Time to try it out

We're now in a position to try our implementation.

__Task:__

- Try running the following code to compute the value of the function $z=x\cdot y+sin(x)$ given $x=0.5$ and $y=4.2$, together with the derivative $\partial z/\partial x$ at that point.

In [27]:
x = DualNumber(0.5, 1.0)
y = DualNumber(4.2, 0.0)

z = x * y + sin(x)
dzdx = z.dvalue

print("z: ", z.value)
print("dz/dx: ", dzdx)
print("z як дуальне число: ", z)

z:  2.579425538604203
dz/dx:  5.077582561890373
z як дуальне число:  2.579425538604203 + 5.077582561890373ε


__Task__: Differentiate the above function with respect to $x$ and write the symbolic derivatives in the following box. Verify the result computed above is correct by plugging-in the values into your symbolic gradient expression.

$z = x\cdot y + \sin(x)$

$\dfrac{\partial z}{\partial x} = y + \cos(x),\qquad \dfrac{\partial z}{\partial y} = x$


Підставимо $x=0.5, y=4.2:$

1) $\dfrac{\partial z}{\partial x}(x,y)=4.2 + 0.8775825618903728 = 5.0775825618903728 ≈ 5.1$

2) $\dfrac{\partial z}{\partial y}(x,y)=0.5$

3) $z(x,y) = 0.5\cdot 4.2 + \sin(0.5)=2.1 + 0.479425538604203 = 2.579425538604203 ≈ 2.6$


Обчислення за допомогою Dualnumber (Forward mode) збігається з символьним результатом до машинної точності.


__Task:__ Now use the code block below to compute the derivative $\partial z/\partial y$ of the above expression (at the same point $x=0.5, y=4.2$ as above) and store the derivative in the variable `dzdy` (just the derivative, not the Dual Number). Verify by hand that the result is correct.

In [28]:
x = DualNumber(0.5, 0.0)
y = DualNumber(4.2, 1.0)

z = x * y + sin(x)
dzdy = z.dvalue

print("z: ", z.value)
print("dz/dy:", dzdy)
print("z як дуальне число: ", z)

z:  2.579425538604203
dz/dy: 0.5
z як дуальне число:  2.579425538604203 + 0.5ε


In [29]:
#Tests
assert dzdy
assert type(dzdy) == float


__Task:__ Finally, use the code block below to experiment and test the other math functions and methods you created.

In [30]:
# Перевірьмо усі функції

u = DualNumber(5.0, 1.0)  # зміна, за якою диференціюємо
v = DualNumber(3.0, 0.0)

print("u + v =", u + v, "     (очікувана похідна = 1)")
print("u - v =", u - v, "     (очікувана похідна = 1)")
print("u * v =", u * v, "     (очікувана похідна = 3)")
print("u / v =", u / v, "     (очікувана похідна = 0.3(3))")
print("u ** v =", u ** v, "   (очікувана похідна = v*u^(v-1) = 75)")
print("u ** u =", u ** u, "   (очікувана похідна: u^u(ln u + 1) = ", 5 ** 5 * (math.log(5) + 1), ")")

u + v = 8.0 + 1.0ε      (очікувана похідна = 1)
u - v = 2.0 + 1.0ε      (очікувана похідна = 1)
u * v = 15.0 + 3.0ε      (очікувана похідна = 3)
u / v = 1.6666666666666667 + 0.3333333333333333ε      (очікувана похідна = 0.3(3))
u ** v = 125.0 + 75.0ε    (очікувана похідна = v*u^(v-1) = 75)
u ** u = 3125.0 + 8154.493476356564ε    (очікувана похідна: u^u(ln u + 1) =  8154.493476356564 )


In [31]:
# Перевіримо метод градієнта для функції кількох змінних:
# y = f(x1, x2, x3) = x1*x2 + sin(x1*x3) + exp(x2)
# буде 3 проходи для трьох похідних

def y(v):
  x1, x2, x3 = v
  return x1 * x2 + sin(x1*x3) + exp(x2)

point = [0.5, 1.2, 2.0]
grad = []

for i in range(len(point)):
  duals = [DualNumber(val, 1.0 if k==i else 0.0) for k, val in enumerate(point)]
  grad.append(y(duals).dvalue)

print("y(point) = ", y([DualNumber(val, 0.0) for val in point]).value)
print("grad(y) = ", grad, "\n")


# порівняння
x1, x2, x3 = point
print("df/dx1 : ", x2 + math.cos(x1 * x3) * x3)
print("df/dx2 : ", x1 + math.exp(x2))
print("df/dx3 : ", math.cos(x1 * x3) * x1)

# ніби усе ок



y(point) =  4.761587907544444
grad(y) =  [2.2806046117362797, 3.8201169227365472, 0.2701511529340699] 

df/dx1 :  2.2806046117362797
df/dx2 :  3.8201169227365472
df/dx3 :  0.2701511529340699
